# 01 — Your first Delta table on Databricks Free Edition

Companion notebook for [Medium post link goes here].

Creates a small fake customers table in `nocompany.demo.customers` 
using Faker. Run cells top to bottom.

Prereqs: Free Edition account, `nocompany` catalog and `demo` schema created.

In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from faker import Faker
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DateType
import random

# Pin the seed so the data is the same every time you run this
fake = Faker('en_IN')   # Indian-flavored fake data, fitting for our context
Faker.seed(42)
random.seed(42)

# NoCompany — a deliberately fake SaaS company for this tutorial series
regions = ['north', 'south', 'east', 'west']
tiers = ['starter', 'growth', 'enterprise']

rows = []
for i in range(1, 13):  # 12 customers
    rows.append(Row(
        customer_id=f"CUST-{i:04d}",
        customer_name=fake.company(),
        region=random.choice(regions),
        signup_date=fake.date_between(start_date='-2y', end_date='today'),
        plan_tier=random.choice(tiers)
    ))

df = spark.createDataFrame(rows)
df.show(truncate=False)

# Save as a Delta table
df.write.mode("overwrite").saveAsTable("nocompany.demo.customers")

print("✅ Table created: nocompany.demo.customers")

+-----------+-----------------------------+------+-----------+----------+
|customer_id|customer_name                |region|signup_date|plan_tier |
+-----------+-----------------------------+------+-----------+----------+
|CUST-0001  |Choudhury, Bakshi and Maharaj|north |2024-11-23 |starter   |
|CUST-0002  |Chaudry Ltd                  |east  |2025-11-20 |starter   |
|CUST-0003  |Chahal, Sami and Balay       |south |2024-06-18 |starter   |
|CUST-0004  |Kaul PLC                     |north |2025-08-11 |enterprise|
|CUST-0005  |Issac, Saini and Kannan      |north |2025-04-21 |enterprise|
|CUST-0006  |Ahuja-Dutta                  |west  |2025-10-20 |starter   |
|CUST-0007  |Mall-Dugal                   |north |2024-11-01 |starter   |
|CUST-0008  |Chaudry-Chanda               |south |2025-03-01 |starter   |
|CUST-0009  |Padmanabhan-Lalla            |north |2026-01-08 |enterprise|
|CUST-0010  |Sharaf, Yadav and Dash       |south |2026-05-09 |enterprise|
|CUST-0011  |Brar-Memon               

In [0]:
%sql
SELECT * FROM nocompany.demo.customers ORDER BY signup_date DESC

customer_id,customer_name,region,signup_date,plan_tier
CUST-0010,"Sharaf, Yadav and Dash",south,2026-05-09,enterprise
CUST-0011,Brar-Memon,west,2026-01-24,starter
CUST-0009,Padmanabhan-Lalla,north,2026-01-08,enterprise
CUST-0002,Chaudry Ltd,east,2025-11-20,starter
CUST-0006,Ahuja-Dutta,west,2025-10-20,starter
CUST-0004,Kaul PLC,north,2025-08-11,enterprise
CUST-0005,"Issac, Saini and Kannan",north,2025-04-21,enterprise
CUST-0008,Chaudry-Chanda,south,2025-03-01,starter
CUST-0001,"Choudhury, Bakshi and Maharaj",north,2024-11-23,starter
CUST-0007,Mall-Dugal,north,2024-11-01,starter
